In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats

In [4]:
DATA_DIR = Path("../Data/raw")
OUTPUT_DIR = Path("../outputs/metrics")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

events = pd.read_csv(
    DATA_DIR / "events.csv",
    parse_dates=["event_time"]
)

assignments = pd.read_csv(
    DATA_DIR / "experiment_assignment.csv"
)

user_metrics = pd.read_csv(
    OUTPUT_DIR / "user_metrics.csv"
)

In [10]:
import pandas as pd
from pathlib import Path

# CHANGE THIS PATH to the folder containing your CSV files
DATA_DIR = Path("../Data/raw")

users = pd.read_csv(DATA_DIR / "users.csv")
products = pd.read_csv(DATA_DIR / "products.csv")
experiment = pd.read_csv(DATA_DIR / "experiment_assignment.csv")
sessions = pd.read_csv(DATA_DIR / "sessions.csv")
events = pd.read_csv(DATA_DIR / "events.csv")
orders = pd.read_csv(DATA_DIR / "orders.csv")

print("Users:", users.shape)
print("Products:", products.shape)
print("Experiment:", experiment.shape)
print("Sessions:", sessions.shape)
print("Events:", events.shape)
print("Orders:", orders.shape)

Users: (120000, 11)
Products: (500, 12)
Experiment: (120000, 4)
Sessions: (659860, 15)
Events: (1974752, 5)
Orders: (44176, 15)


In [11]:
events["is_homepage"] = (
    events["event_name"]
    .eq("Homepage")
)

events["is_search"] = (
    events["event_name"]
    .eq("Search")
)

events["is_product_view"] = (
    events["event_name"]
    .eq("Product_View")
)

events["is_add_cart"] = (
    events["event_name"]
    .eq("Add_Cart")
)

events["is_checkout"] = (
    events["event_name"]
    .eq("Checkout")
)

events["is_shipping"] = (
    events["event_name"]
    .eq("Shipping")
)

events["is_payment"] = (
    events["event_name"]
    .isin([
        "Payment",
        "One_Click_Payment"
    ])
)

events["is_purchase"] = (
    events["event_name"]
    .isin([
        "Purchase",
        "Order_Confirmed"
    ])
)

In [12]:
events = events.merge(
    sessions[["session_id", "user_id"]],
    on="session_id",
    how="left",
    validate="many_to_one"
)

In [13]:
events["is_homepage"] = (
    events["event_name"] == "Homepage"
).astype(int)

events["is_search"] = (
    events["event_name"] == "Search"
).astype(int)

events["is_product_view"] = (
    events["event_name"] == "Product_View"
).astype(int)

events["is_add_cart"] = (
    events["event_name"] == "Add_to_Cart"
).astype(int)

events["is_checkout"] = (
    events["event_name"] == "Checkout"
).astype(int)

events["is_shipping"] = (
    events["event_name"] == "Shipping"
).astype(int)

events["is_payment"] = (
    events["event_name"].isin(
        ["Payment", "One_Click_Payment"]
    )
).astype(int)

events["is_purchase"] = (
    events["event_name"] == "Purchase"
).astype(int)

In [14]:
funnel = pd.DataFrame({
    "stage": [
        "Homepage",
        "Search",
        "Product View",
        "Add Cart",
        "Checkout",
        "Shipping",
        "Payment",
        "Purchase"
    ],

    "users": [
        events.loc[
            events["is_homepage"],
            "user_id"
        ].nunique(),

        events.loc[
            events["is_search"],
            "user_id"
        ].nunique(),

        events.loc[
            events["is_product_view"],
            "user_id"
        ].nunique(),

        events.loc[
            events["is_add_cart"],
            "user_id"
        ].nunique(),

        events.loc[
            events["is_checkout"],
            "user_id"
        ].nunique(),

        events.loc[
            events["is_shipping"],
            "user_id"
        ].nunique(),

        events.loc[
            events["is_payment"],
            "user_id"
        ].nunique(),

        events.loc[
            events["is_purchase"],
            "user_id"
        ].nunique()
    ]
})

funnel

,stage,users
0,Homepage,2
1,Search,2
2,Product View,2
3,Add Cart,1
4,Checkout,2
5,Shipping,2
6,Payment,2
7,Purchase,2


In [15]:
funnel["stage_conversion"] = (
    funnel["users"]
    /
    funnel.iloc[0]["users"]
)

In [16]:
funnel["previous_stage_users"] = (
    funnel["users"].shift(1)
)

funnel["stage_to_stage_conversion"] = (
    funnel["users"]
    /
    funnel["previous_stage_users"]
)

funnel.loc[
    funnel.index == 0,
    "stage_to_stage_conversion"
] = 1

In [17]:
funnel["drop_off"] = (
    1 -
    funnel["stage_to_stage_conversion"]
)

In [18]:
funnel.to_csv(
    OUTPUT_DIR / "funnel_metrics.csv",
    index=False
)

In [23]:
def calculate_funnel(data):

    return {
        "Homepage":
            data.loc[
                data["is_homepage"] == 1,
                "user_id"
            ].nunique(),

        "Search":
            data.loc[
                data["is_search"] == 1,
                "user_id"
            ].nunique(),

        "Product View":
            data.loc[
                data["is_product_view"] == 1,
                "user_id"
            ].nunique(),

        "Add Cart":
            data.loc[
                data["is_add_cart"] == 1,
                "user_id"
            ].nunique(),

        "Checkout":
            data.loc[
                data["is_checkout"] == 1,
                "user_id"
            ].nunique(),

        "Shipping":
            data.loc[
                data["is_shipping"] == 1,
                "user_id"
            ].nunique(),

        "Payment":
            data.loc[
                data["is_payment"] == 1,
                "user_id"
            ].nunique(),

        "Purchase":
            data.loc[
                data["is_purchase"] == 1,
                "user_id"
            ].nunique()
    }

In [24]:
control_users_ids = set(
    user_metrics.loc[
        user_metrics["group"] == "Control",
        "user_id"
    ]
)

treatment_users_ids = set(
    user_metrics.loc[
        user_metrics["group"] == "Treatment",
        "user_id"
    ]
)

In [25]:
control_events = events[
    events["user_id"].isin(control_users_ids)
]

treatment_events = events[
    events["user_id"].isin(treatment_users_ids)
]

In [26]:
control_funnel = calculate_funnel(
    control_events
)

treatment_funnel = calculate_funnel(
    treatment_events
)

In [27]:
funnel_comparison = pd.DataFrame({
    "stage": control_funnel.keys(),

    "control_users":
        control_funnel.values(),

    "treatment_users":
        treatment_funnel.values()
})

In [28]:
funnel_comparison["control_conversion"] = (
    funnel_comparison["control_users"]
    /
    funnel_comparison.iloc[0]["control_users"]
)

funnel_comparison["treatment_conversion"] = (
    funnel_comparison["treatment_users"]
    /
    funnel_comparison.iloc[0]["treatment_users"]
)

In [29]:
funnel_comparison["uplift"] = (
    funnel_comparison["treatment_conversion"]
    -
    funnel_comparison["control_conversion"]
)

In [30]:
funnel_comparison.to_csv(
    OUTPUT_DIR / "funnel_comparison.csv",
    index=False
)

In [31]:
group_counts = (
    assignments["group"]
    .value_counts()
)

group_counts

group
Control      60000
Treatment    60000
Name: count, dtype: int64

In [32]:
control_share = (
    group_counts["Control"]
    /
    group_counts.sum()
)

treatment_share = (
    group_counts["Treatment"]
    /
    group_counts.sum()
)

print(control_share)
print(treatment_share)

0.5
0.5


In [33]:
observed = [
    group_counts["Control"],
    group_counts["Treatment"]
]

expected = [
    sum(observed) / 2,
    sum(observed) / 2
]

chi2_srm, p_srm = stats.chisquare(
    observed,
    f_exp=expected
)

print("SRM p-value:", p_srm)

SRM p-value: 1.0
